# Feature Preprocessing Pipeline

This notebook implements all preprocessing decisions made after the EDA review, covering four concerns:

1. **Join key construction** — replace the broken `county_fips` key with a stable composite identifier
2. **Collinearity** — drop redundant broadband features
3. **Missing value imputation** — per-feature strategy based on missingness mechanism
4. **Feature transforms** — `log1p` for right-skewed features

**Input:** `data_revealed/03_tables/county_final_table_clean.csv`  
**Output:** `data_revealed/04_tables/county_preprocessed.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

PROJECT_ROOT = Path().resolve().parent
os.chdir(PROJECT_ROOT)
print("Working dir:", os.getcwd())

df = pd.read_csv("data_revealed/03_tables/county_final_table_clean.csv")
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} cols")

Working dir: /Users/rx/Documents/Self-Development/github-project/data-center-siting-analysis
Loaded: 3149 rows x 41 cols


## 1. Join Key Construction

Three counties — Middletown County CT, Athens-Clarke County GA, and Loudoun County VA — are
consolidated city-county governments that were assigned a placeholder `county_fips` value (`0<NA>`)
in the upstream reference table. Their feature columns are fully populated, so they are usable
for modeling, but `county_fips` cannot serve as a unique row key.

Since geographic visualization is out of scope, we replace `county_fips` as the primary identifier
with a composite `county_key = state || county`. By US county naming convention, the
(state, county) pair is always unique, and both fields have 0% missingness across all 3,149 rows.

In [2]:
df["county_key"] = df["state"] + "||" + df["county"]

assert df["county_key"].duplicated().sum() == 0, "county_key is not unique — investigate!"
print(f"county_key created — {df['county_key'].nunique()} unique values")

# Confirm the three formerly-problematic counties now have a clean key
problem_fips_mask = df["county_fips"].astype(str).str.startswith("0")
print("\nFormerly-placeholder FIPS rows:")
print(df.loc[problem_fips_mask, ["county_key", "county_fips", "state", "county"]])

county_key created — 3149 unique values

Formerly-placeholder FIPS rows:
                         county_key county_fips        state  \
0           Alabama||Autauga County       01001      Alabama   
1           Alabama||Baldwin County       01003      Alabama   
2           Alabama||Barbour County       01005      Alabama   
3              Alabama||Bibb County       01007      Alabama   
4            Alabama||Blount County       01009      Alabama   
..                              ...         ...          ...   
315     Connecticut||Tolland County       09013  Connecticut   
316     Connecticut||Windham County       09015  Connecticut   
317  Connecticut||Middletown County       0<NA>  Connecticut   
318   Georgia||Athens-Clarke County       0<NA>      Georgia   
319         Virginia||Loudon County       0<NA>     Virginia   

                   county  
0          Autauga County  
1          Baldwin County  
2          Barbour County  
3             Bibb County  
4           Blount

## 2. Drop Redundant Features (Collinearity)

The FCC broadband table contributes six coverage-rate features at different speed/technology
thresholds. EDA revealed two problems:

- **`any_tech_100_20_coverage`** is near-constant (std ≈ 0.04, skew = −21): virtually every US
  county already has ≥100/20 Mbps coverage somewhere. It carries almost no discriminating signal
  and is dropped unconditionally.
- The remaining five coverage features (`cable_fiber_100_20_coverage`, `fiber_100_20_coverage`,
  `high_speed_wired_coverage`, `any_tech_1000_100_coverage`, `fiber_availability`) are highly
  inter-correlated, all sourced from the same FCC filing at different thresholds.

We retain exactly two features that capture distinct infrastructure dimensions:
- **`fiber_availability`**: whether fiber infrastructure physically exists (supply-side)
- **`any_tech_1000_100_coverage`**: whether gigabit-class service is actually delivered (demand-side)

The other three redundant coverage features are dropped.

In [3]:
DROP_BROADBAND = [
    "any_tech_100_20_coverage",     # near-constant (std ≈ 0.04)
    "cable_fiber_100_20_coverage",  # redundant with fiber_availability
    "fiber_100_20_coverage",        # redundant with fiber_availability
    "high_speed_wired_coverage",    # redundant with any_tech_1000_100_coverage
]

df = df.drop(columns=DROP_BROADBAND)
print(f"Dropped {len(DROP_BROADBAND)} redundant broadband features.")
print(f"Remaining columns: {df.shape[1]}")

Dropped 4 redundant broadband features.
Remaining columns: 38


## 2b. Suppression Floor Indicators

Three features have a massive pile-up at a single exact value — a classic sign of **BEA/EIA
statistical disclosure suppression**: when a county's true value falls below a confidentiality
threshold, the agency replaces it with a floor constant rather than reporting NaN.

| Feature | Floor value | Counties at floor |
|---|---|---|
| `epg_natural_gas` | 5.0 | 76.2% |
| `clean_energy_jobs` | 15.0 | 47.5% |
| `grid_infrastructure_jobs` | 20.0 | 15.7% |

These floor values are not real measurements. To make the censoring explicit and give the model
a clean binary signal, we add `_above_floor` indicator columns **before** any log1p transform
(so the flag is created from the raw original values):

- `1` = county has a real value above the threshold (data is informative)
- `0` = county is at the floor (value is suppressed/below threshold; continuous value is uninformative)

In [4]:
FLOOR_SPECS = {
    "epg_natural_gas": 5.0,
    "clean_energy_jobs": 15.0,
    "grid_infrastructure_jobs": 20.0,
}

for col, floor_val in FLOOR_SPECS.items():
    flag_col = col + "_above_floor"
    df[flag_col] = (df[col] > floor_val).astype(int)
    n_above = df[flag_col].sum()
    print(f"{flag_col}: {n_above} counties above floor ({n_above/len(df):.1%})")

epg_natural_gas_above_floor: 743 counties above floor (23.6%)
clean_energy_jobs_above_floor: 1644 counties above floor (52.2%)
grid_infrastructure_jobs_above_floor: 2647 counties above floor (84.1%)


## 3. Missing Value Imputation

Strategy is determined by the likely missingness mechanism for each feature:

| Mechanism | Treatment |
|---|---|
| **Structural zero** — phenomenon doesn't apply to those counties | Impute **0** |
| **Structurally missing + informative** — absence itself is predictive | Add binary flag, then impute median |
| **Recording gaps (<2%)** — small fraction of counties with incomplete source records | Impute **median** of observed values |

The target (`num_datacenters`) is handled separately in Section 4 — rows with a missing target are dropped entirely rather than imputed.

In [5]:
# ── 3a. Hurricane risk: structural zeros
# The NRI does not compute a hurricane risk index for inland counties — their risk is genuinely
# zero. Imputing 0 is the correct domain value, not an approximation.
n_hurricane = df["hurricane_risk_index_value"].isna().sum()
df["hurricane_risk_index_value"] = df["hurricane_risk_index_value"].fillna(0)
print(f"hurricane_risk_index_value: {n_hurricane} nulls -> filled with 0")

hurricane_risk_index_value: 920 nulls -> filled with 0


In [6]:
# ── 3b. Land value: fix erroneous negatives, then add missingness flag + median imputation
#
# "Standardized" in the column name means per-1/4-acre unit normalization, not z-score.
# Values are in dollars (median ~$90k). 23 rural counties (~1%) carry negative values —
# source-data artifacts from negative net adjustments in county assessor parcel records.
# Land cannot have a negative market value, so these are replaced with NaN before imputation.
#
# The missingness flag is created after nulling the negatives, so it captures both
# the structural missingness (~28%, rural/remote counties with no assessment) and the
# corrected erroneous rows (~1%). The median is computed on the corrected non-null values.

n_neg = (df["land_value_1_4_acre_standardized"] < 0).sum()
df.loc[df["land_value_1_4_acre_standardized"] < 0, "land_value_1_4_acre_standardized"] = np.nan
print(f"land_value_1_4_acre_standardized: {n_neg} negative values converted to NaN")

land_median = df["land_value_1_4_acre_standardized"].median()
df["land_value_missing"] = df["land_value_1_4_acre_standardized"].isna().astype(int)
df["land_value_1_4_acre_standardized"] = df["land_value_1_4_acre_standardized"].fillna(land_median)
print(f"land_value_missing flag: {df['land_value_missing'].sum()} counties flagged (structural missing + corrected errors)")
print(f"land_value_1_4_acre_standardized imputed with median = {land_median:.2f}")

land_value_1_4_acre_standardized: 23 negative values converted to NaN
land_value_missing flag: 905 counties flagged (structural missing + corrected errors)
land_value_1_4_acre_standardized imputed with median = 91250.00


In [7]:
# ── 3c. Ice storm risk: structural zeros
# NRI does not rate warm/southern counties for ice storm risk — absence means no risk.
n_ice = df["ice_storm_risk_index_value"].isna().sum()
df["ice_storm_risk_index_value"] = df["ice_storm_risk_index_value"].fillna(0)
print(f"ice_storm_risk_index_value: {n_ice} nulls -> filled with 0")

ice_storm_risk_index_value: 147 nulls -> filled with 0


In [8]:
# ── 3d. Information-sector wage: structural zeros
# Counties with no IT-sector employment have no wage to report.
# A 0 correctly encodes "this sector does not exist here".
n_wage = df["wage_information"].isna().sum()
df["wage_information"] = df["wage_information"].fillna(0)
print(f"wage_information: {n_wage} nulls -> filled with 0")

wage_information: 104 nulls -> filled with 0


In [9]:
# ── 3e. Dock presence: structural zeros
# Missing means no dock infrastructure — same as a zero value.
n_dock = df["dock_presence"].isna().sum()
df["dock_presence"] = df["dock_presence"].fillna(0)
print(f"dock_presence: {n_dock} nulls -> filled with 0")

dock_presence: 56 nulls -> filled with 0


In [10]:
# ── 3f. Lightning & landslide risk: structural zeros (NRI geographic coverage gaps)
for col in ["lightning_risk_index_value", "landslide_risk_index_value"]:
    n = df[col].isna().sum()
    df[col] = df[col].fillna(0)
    print(f"{col}: {n} nulls -> filled with 0")

lightning_risk_index_value: 41 nulls -> filled with 0
landslide_risk_index_value: 36 nulls -> filled with 0


In [11]:
# ── 3g. Transport connectivity: median imputation (recording gaps, ~1.78%)
# These are continuous index/intensity scores — median is appropriate for small gaps.
for col in ["infrastructure_quality", "rail_intensity", "air_connectivity"]:
    med = df[col].median()
    n = df[col].isna().sum()
    df[col] = df[col].fillna(med)
    print(f"{col}: {n} nulls -> filled with median={med:.4f}")

infrastructure_quality: 61 nulls -> filled with median=0.6786
rail_intensity: 56 nulls -> filled with median=3.6109
air_connectivity: 56 nulls -> filled with median=0.5306


In [12]:
# ── 3h. Electricity prices: fix implausible values + median imputation
#
# industrial_price has source data artifacts at the low end:
# - 15 counties with exact 0.0 values (no industrial electricity market / data errors)
# - 2 counties with near-zero floats (~4e-6 $/kWh, stored as tiny positives, not caught by == 0)
# - 6 additional counties with values < 0.01 $/kWh (< 1 cent/kWh)
# US industrial rates range $0.04–0.15/kWh; anything below $0.01/kWh is implausible.
# We apply a minimum plausible threshold: values below 0.01 $/kWh are treated as missing.
# commercial_price has no such artifacts but goes through the same loop for consistency.

PRICE_MIN_THRESHOLD = 0.01  # $/kWh — below this is implausible for US electricity prices

for col in ["commercial_price", "industrial_price"]:
    n_bad = (df[col] < PRICE_MIN_THRESHOLD).sum()
    if n_bad > 0:
        df.loc[df[col] < PRICE_MIN_THRESHOLD, col] = np.nan
        print(f"{col}: {n_bad} implausible values (< {PRICE_MIN_THRESHOLD} $/kWh) -> NaN")
    med = df[col].median()
    n_null = df[col].isna().sum()
    df[col] = df[col].fillna(med)
    print(f"{col}: {n_null} nulls -> filled with median={med:.4f}")

commercial_price: 45 nulls -> filled with median=0.1106
industrial_price: 23 implausible values (< 0.01 $/kWh) -> NaN
industrial_price: 68 nulls -> filled with median=0.0752


In [13]:
# ── 3i. Remaining wage features: median imputation (<1%)
for col in ["wage_prof_business", "wage_trade_transport_utilities"]:
    med = df[col].median()
    n = df[col].isna().sum()
    df[col] = df[col].fillna(med)
    print(f"{col}: {n} nulls -> filled with median={med:.4f}")

wage_prof_business: 19 nulls -> filled with median=1078.0000
wage_trade_transport_utilities: 14 nulls -> filled with median=865.0000


In [14]:
# ── 3j. All remaining features with <0.5% missingness: median imputation
# Covers: remaining NRI indices, broadband coverage rates, energy/job counts.
# At this rate, median imputation introduces negligible bias.
NON_FEATURE_COLS = {"num_datacenters", "county_fips", "county_key", "state", "county"}
remaining_null_cols = [
    c for c in df.columns
    if df[c].isna().any() and c not in NON_FEATURE_COLS
]
for col in remaining_null_cols:
    med = df[col].median()
    n = df[col].isna().sum()
    df[col] = df[col].fillna(med)
    print(f"{col}: {n} nulls -> median={med:.4f}")

# Verify: no feature-column nulls remain
feature_nulls = df.drop(columns=list(NON_FEATURE_COLS)).isnull().sum()
remaining = feature_nulls[feature_nulls > 0]
if remaining.empty:
    print("\nAll feature nulls resolved.")
else:
    print("\nStill null:", remaining)

epg_natural_gas: 8 nulls -> median=5.0000
clean_energy_jobs: 8 nulls -> median=20.7635
grid_infrastructure_jobs: 8 nulls -> median=82.4784
any_tech_1000_100_coverage: 8 nulls -> median=0.3197
fiber_availability: 8 nulls -> median=0.2605
community_resilience_value: 6 nulls -> median=2.5978
community_risk_factor_value: 6 nulls -> median=1.1673
cold_wave_risk_index_value: 6 nulls -> median=64326.1906
drought_risk_index_value: 6 nulls -> median=26816.0260
earthquake_risk_index_value: 6 nulls -> median=154851.7042
hail_risk_index_value: 6 nulls -> median=107363.4120
heat_wave_risk_index_value: 6 nulls -> median=86890.5205
riverine_flooding_risk_index_value: 6 nulls -> median=463730.3715
strong_wind_risk_index_value: 6 nulls -> median=384767.9927
tornado_risk_index_value: 6 nulls -> median=1181368.2730
wildfire_risk_index_value: 6 nulls -> median=39993.2513
winter_weather_risk_index_value: 6 nulls -> median=60578.9159

All feature nulls resolved.


## 4. Drop Rows with Missing Target

`num_datacenters` has 11 rows (0.35%) where the ZIP-to-county allocation failed, leaving no
reliable estimate. Imputing the target variable would introduce noise into the label itself,
which is worse than losing 11 rows out of 3,149. These rows are dropped.

In [15]:
before = len(df)
df = df.dropna(subset=["num_datacenters"])
after = len(df)
print(f"Dropped {before - after} rows with missing target. Remaining: {after}")

Dropped 11 rows with missing target. Remaining: 3138


## 5. Feature Transforms (log1p)

Features with high positive skew compress poorly during training — outliers dominate the loss
gradient and can inflate feature importance. `log1p(x) = log(1 + x)` is appropriate here because:
- Maps `x = 0` → `0`, preserving structural zeros common in hazard indices and job counts
- Compresses the right tail without requiring strictly positive values
- Is invertible (original-scale values are recoverable)

**Transformed (|skew| > 1, confirmed non-negative after imputation):**
- All 14 NRI hazard risk indices
- Energy/employment counts: `grid_infrastructure_jobs`, `clean_energy_jobs`, `epg_natural_gas`
- Wages: `wage_information`, `wage_prof_business`, `wage_trade_transport_utilities`
- Cost/value: `commercial_price`, `industrial_price`, `land_value_1_4_acre_standardized`
  (negative values corrected in step 3b; zeros corrected in step 3h — fully non-negative)
- Connectivity: `air_connectivity`

**Not transformed:**
- `dock_presence`: confirmed binary (only values 0 and 1 in raw data). log1p(1) = 0.693 is
  a trivial rescaling with no modeling benefit — kept as 0/1.
- Coverage rates (`fiber_availability`, `any_tech_1000_100_coverage`): bounded [0, 1], skew < 1 — standardize at model time
- Composite NRI scores (`community_resilience_value`, `community_risk_factor_value`): already normalized
- `infrastructure_quality`, `rail_intensity`: low or mildly left-skewed
- Binary/sparse: `has_policy_signal`, `policy_direction_score`, `land_value_missing`,
  `epg_natural_gas_above_floor`, `clean_energy_jobs_above_floor`, `grid_infrastructure_jobs_above_floor`

In [16]:
LOG1P_COLS = [
    # NRI hazard risk indices (14)
    "cold_wave_risk_index_value",
    "drought_risk_index_value",
    "earthquake_risk_index_value",
    "hail_risk_index_value",
    "heat_wave_risk_index_value",
    "hurricane_risk_index_value",
    "ice_storm_risk_index_value",
    "landslide_risk_index_value",
    "lightning_risk_index_value",
    "riverine_flooding_risk_index_value",
    "strong_wind_risk_index_value",
    "tornado_risk_index_value",
    "wildfire_risk_index_value",
    "winter_weather_risk_index_value",
    # Energy / employment
    "grid_infrastructure_jobs",
    "clean_energy_jobs",
    "epg_natural_gas",
    # Wages
    "wage_information",
    "wage_prof_business",
    "wage_trade_transport_utilities",
    # Cost / value
    "commercial_price",
    "industrial_price",
    "land_value_1_4_acre_standardized",
    # Connectivity
    "air_connectivity",
    # dock_presence excluded: confirmed binary (0/1), log1p offers no benefit
]

# Safety check: log1p requires non-negative values
for col in LOG1P_COLS:
    min_val = df[col].min()
    assert min_val >= 0, f"{col} has negative min={min_val:.4f}; log1p cannot be applied"

skew_before = df[LOG1P_COLS].skew()
df[LOG1P_COLS] = np.log1p(df[LOG1P_COLS])
skew_after = df[LOG1P_COLS].skew()

comparison = pd.DataFrame({"before": skew_before, "after": skew_after})
comparison["reduction"] = comparison["before"] - comparison["after"]
print(f"log1p applied to {len(LOG1P_COLS)} features.\n")
print(comparison.round(2).sort_values("before", ascending=False).to_string())

log1p applied to 24 features.

                                    before  after  reduction
cold_wave_risk_index_value           39.52  -0.72      40.24
earthquake_risk_index_value          36.50   0.10      36.39
riverine_flooding_risk_index_value   36.26  -2.24      38.50
wildfire_risk_index_value            31.23   0.39      30.84
drought_risk_index_value             25.46  -0.97      26.44
heat_wave_risk_index_value           20.95  -1.16      22.11
tornado_risk_index_value             20.47  -1.79      22.26
grid_infrastructure_jobs             19.60   0.87      18.73
land_value_1_4_acre_standardized     19.50  -0.26      19.77
epg_natural_gas                      18.40   2.27      16.13
hail_risk_index_value                17.91  -1.13      19.04
hurricane_risk_index_value           17.88  -0.43      18.31
landslide_risk_index_value           14.09  -2.62      16.71
ice_storm_risk_index_value           12.71  -1.84      14.55
clean_energy_jobs                    12.30   1.77     

## 6. Final Column Order & Save

Identifiers first, features in the middle, target last.

In [17]:
import os
os.makedirs("data_revealed/04_tables", exist_ok=True)

ID_COLS = ["county_key", "county_fips", "state", "county"]
TARGET_COL = ["num_datacenters"]
feature_cols = [c for c in df.columns if c not in ID_COLS + TARGET_COL]
df = df[ID_COLS + feature_cols + TARGET_COL]

out_path = "data_revealed/04_tables/county_preprocessed.csv"
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"\nAll columns ({df.shape[1]}):")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2d}. {c}")

Saved: data_revealed/04_tables/county_preprocessed.csv
Shape: 3138 rows x 42 cols

All columns (42):
   1. county_key
   2. county_fips
   3. state
   4. county
   5. epg_natural_gas
   6. clean_energy_jobs
   7. grid_infrastructure_jobs
   8. any_tech_1000_100_coverage
   9. fiber_availability
  10. land_value_1_4_acre_standardized
  11. commercial_price
  12. industrial_price
  13. air_connectivity
  14. rail_intensity
  15. infrastructure_quality
  16. dock_presence
  17. community_resilience_value
  18. community_risk_factor_value
  19. cold_wave_risk_index_value
  20. drought_risk_index_value
  21. earthquake_risk_index_value
  22. hail_risk_index_value
  23. heat_wave_risk_index_value
  24. hurricane_risk_index_value
  25. ice_storm_risk_index_value
  26. landslide_risk_index_value
  27. lightning_risk_index_value
  28. riverine_flooding_risk_index_value
  29. strong_wind_risk_index_value
  30. tornado_risk_index_value
  31. wildfire_risk_index_value
  32. winter_weather_risk_ind

In [18]:
# Final sanity check
print("=== Final sanity check ===")
print(f"Rows: {len(df)}")
print(f"Any nulls remaining: {df.isnull().any().any()}")
print(f"Target zeros: {(df['num_datacenters'] == 0).sum()} ({(df['num_datacenters'] == 0).mean():.1%})")
print(f"land_value_missing counties: {df['land_value_missing'].sum()}")
print(f"\nSample (first 3 rows, identifier + target):")
print(df[["county_key", "num_datacenters"]].head(3))

=== Final sanity check ===
Rows: 3138
Any nulls remaining: False
Target zeros: 2371 (75.6%)
land_value_missing counties: 895

Sample (first 3 rows, identifier + target):
                county_key  num_datacenters
0  Alabama||Autauga County              0.0
1  Alabama||Baldwin County              0.0
2  Alabama||Barbour County              0.0
